# 05 - Batch and Online Evaluation

This notebook turns evaluation into an operational system.

You will:

1. Distinguish a curated benchmark from batch historical evaluation.
2. Build a calibrated CI quality gate.
3. Configure sampled online evaluation.
4. Connect evaluation results to CloudWatch logs, metrics, and alarms.

**Estimated time:** 35-50 minutes  
**Creates AWS resources:** Optional batch jobs and online evaluation configuration.  
**Feature status:** Batch evaluation is public preview as of August 14, 2026 and does not currently emit CloudTrail events.

## 1. Choose the right execution mode

| Need | Mechanism |
|---|---|
| Investigate one recent session | `agentcore run eval --session-id ...` |
| Run a curated benchmark that invokes the agent | Dataset runner or managed dataset |
| Re-score many existing sessions asynchronously | Batch evaluation |
| Continuously sample live traffic | Online evaluation |

"Online" means asynchronous continuous evaluation of sampled live traces. It does not mean the judge blocks the user response in real time.

## 2. Batch evaluation over historical sessions

Batch evaluation is useful for backfills, score-drift analysis, and large pre-release checks against already collected traffic.

In [ ]:
import subprocess
from pathlib import Path

MODULE_ROOT = Path.cwd()
RUN_BATCH_EVALUATION = False

if RUN_BATCH_EVALUATION:
    subprocess.run(
        [
            "agentcore",
            "run",
            "batch-evaluation",
            "--runtime",
            "CityAnalyst",
            "--evaluator",
            "Builtin.Helpfulness",
            "Builtin.GoalSuccessRate",
            "--lookback-days",
            "1",
            "--name",
            "CityAnalystRegression",
            "--wait",
        ],
        cwd=MODULE_ROOT,
        check=True,
    )
else:
    print("Set RUN_BATCH_EVALUATION=True after generating several sessions.")

Batch jobs write aggregate status and per-session results. Preserve:

- evaluator version or ARN
- runtime version or endpoint
- dataset or session source
- time range
- score distribution, not only the mean
- failed and ignored session counts

## 3. A CI quality gate

A quality gate should fail closed when:

- the evaluation command fails
- no sessions are found
- evaluator results contain errors
- a calibrated metric crosses its decision threshold

The threshold below is an example placeholder. Replace it only after validating the evaluator on human-labeled data.

In [ ]:
import json
import subprocess


def run_quality_gate(
    runtime_name: str,
    evaluator_id: str,
    threshold: float,
    lookback_days: int = 1,
) -> dict:
    completed = subprocess.run(
        [
            "agentcore",
            "run",
            "eval",
            "--runtime",
            runtime_name,
            "--evaluator",
            evaluator_id,
            "--days",
            str(lookback_days),
            "--json",
        ],
        cwd=MODULE_ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    payload = json.loads(completed.stdout)
    run = payload.get("run", payload)
    result_rows = run.get("results", [])
    if not result_rows:
        raise RuntimeError("Quality gate found no evaluation results.")

    matching = [
        item
        for item in result_rows
        if item.get("evaluator") == evaluator_id
    ]
    if not matching:
        raise RuntimeError(f"No result found for {evaluator_id}.")

    score = matching[0].get("aggregateScore")
    if score is None:
        raise RuntimeError("Evaluation result did not include aggregateScore.")
    if score < threshold:
        raise RuntimeError(
            f"Quality gate failed: {score:.3f} < {threshold:.3f}"
        )
    return {"score": score, "threshold": threshold, "passed": True}


CALIBRATED_THRESHOLD = 0.70
RUN_QUALITY_GATE = False

if RUN_QUALITY_GATE:
    print(
        run_quality_gate(
            runtime_name="CityAnalyst",
            evaluator_id="Builtin.Helpfulness",
            threshold=CALIBRATED_THRESHOLD,
        )
    )
else:
    print("Calibrate the threshold, then set RUN_QUALITY_GATE=True.")

In CI, invoke representative integration tests first, poll until their traces are visible, then run the gate. Do not use a long fixed delay copied from an old notebook.

## 4. Configure online evaluation

For controlled workshop traffic, `100` means every invocation. In production, start around `1-5` percent unless risk, regulation, or very low traffic justifies a higher rate.

In [ ]:
import json

ONLINE_CONFIG_NAME = "CityQualityMonitor"
CREATE_ONLINE_CONFIG = False
WORKSHOP_SAMPLING_PERCENTAGE = 100

project_config = json.loads(
    (MODULE_ROOT / "agentcore" / "agentcore.json").read_text()
)
existing_online = {
    item["name"]
    for item in project_config.get("onlineEvalConfigs", [])
}

if CREATE_ONLINE_CONFIG and ONLINE_CONFIG_NAME not in existing_online:
    subprocess.run(
        [
            "agentcore",
            "add",
            "online-eval",
            "--name",
            ONLINE_CONFIG_NAME,
            "--runtime",
            "CityAnalyst",
            "--evaluator",
            "Builtin.Helpfulness",
            "Builtin.GoalSuccessRate",
            "Builtin.ToolSelectionAccuracy",
            "--sampling-rate",
            str(WORKSHOP_SAMPLING_PERCENTAGE),
            "--enable-on-create",
        ],
        cwd=MODULE_ROOT,
        check=True,
    )
    print("Online config added locally. Review agentcore.json, then deploy.")
else:
    print("Set CREATE_ONLINE_CONFIG=True to add the workshop monitor.")

Review the diff before deployment:

```bash
agentcore deploy --diff
agentcore deploy
```

Configuration names use letters, numbers, and underscores. Avoid hyphens when the service naming rule does not permit them.

## 5. Generate controlled traffic

Run this only after the online configuration is deployed and active.

In [ ]:
from src.workshop_utils import RuntimeInvoker, load_runtime_info, make_session_id

GENERATE_TRAFFIC = False
prompts = [
    "What are the workshop facts for Seattle, WA?",
    "Compare Boston, MA with Miami, FL.",
    "Calculate density for 400000 people and 80 square miles.",
    "What is the population of Atlantis, CA?",
    "Hello. What can you help me with?",
]

if GENERATE_TRAFFIC:
    runtime = load_runtime_info()
    runtime_invoker = RuntimeInvoker(runtime)
    for prompt in prompts:
        response = runtime_invoker.invoke(
            prompt=prompt,
            session_id=make_session_id("online"),
        )
        print(prompt)
        print(json.dumps(response, indent=2))
else:
    print("Deploy the online config, then set GENERATE_TRAFFIC=True.")

## 6. Observe and control the monitor

Online evaluation results are written to CloudWatch and can be streamed with:

```bash
agentcore logs evals --runtime CityAnalyst --since 1h
```

Pause and resume without deleting the config:

```bash
agentcore pause online-eval CityQualityMonitor
agentcore resume online-eval CityQualityMonitor
```

Use pause during incident response, unexpected cost growth, evaluator maintenance, or when sensitive traffic should not be evaluated.

## 7. Discover metrics before creating alarms

CloudWatch namespace and metric names are case-sensitive and may evolve. Discover what the deployed resources emit instead of copying a hard-coded metric name.

In [ ]:
import boto3
from botocore.config import Config
from src.workshop_utils import load_runtime_info

runtime = load_runtime_info()
cloudwatch = boto3.client(
    "cloudwatch",
    region_name=runtime.region,
    config=Config(
        retries={"total_max_attempts": 5, "mode": "adaptive"},
        connect_timeout=5,
        read_timeout=30,
    ),
)

candidate_namespaces = ["Bedrock-AgentCore", "Bedrock-Agentcore"]
discovered = {}
for namespace in candidate_namespaces:
    paginator = cloudwatch.get_paginator("list_metrics")
    metrics = []
    for page in paginator.paginate(Namespace=namespace):
        metrics.extend(page.get("Metrics", []))
    if metrics:
        discovered[namespace] = sorted(
            {item["MetricName"] for item in metrics}
        )

discovered

Build alarms around calibrated aggregate metrics and operational failure signals. A quality alarm should include enough context to investigate the associated sessions and traces, but should not publish raw sensitive prompts to an unrestricted SNS topic.

## 8. Production checklist

- sample according to risk, volume, and cost
- encrypt result log groups and SNS topics with KMS where required
- set CloudWatch retention
- restrict access to traces and explanations
- scrub PII before telemetry export
- account for cross-region judge inference in data-residency reviews
- monitor evaluator errors separately from low agent scores
- version evaluators and thresholds with the application
- preserve a small human-review sample to detect judge drift

## 9. Checkpoint

You can now connect:

- curated pre-release regression
- historical batch scoring
- CI decisions
- sampled production monitoring

Continue to [06 - Simulation and Optimization](06-simulation-and-optimization.ipynb) to generate harder conversations, find failure patterns, and test improvements.